# Pandas Part 2: Filtering, Aggregation & Groupby

Once you can load and inspect a DataFrame, the next essential skills are filtering rows to focus on relevant subsets, aggregating data to find patterns, and creating new features from existing columns. These operations appear in almost every data preparation workflow — understanding them well makes the difference between code that works and code that's readable and maintainable.

<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>What you will learn</b>

- Filter rows with boolean conditions (single and combined)
- Sort data by one or multiple columns
- Group data and compute aggregations with <code>groupby</code>
- Create new columns from existing ones

<b>Dataset:</b> AI/ML Job Salaries — global salary data for data science roles (2020-2024).

</div>

---
## Step 1: Setup and Loading Data

In [1]:
import pandas as pd

df = pd.read_csv('salaries.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (73148, 11)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M
1,2025,SE,FT,Data Product Owner,110000,USD,110000,US,0,US,M
2,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M


---
## Step 2: Boolean Filtering

Boolean filtering works by creating a mask — a Series of True/False values — and applying it to select rows.

In [2]:
# Example of boolean masking
df['experience_level'] == 'SE'

0         True
1         True
2         True
3         True
4         True
         ...  
73143     True
73144    False
73145    False
73146    False
73147     True
Name: experience_level, Length: 73148, dtype: bool

In [3]:
# Single condition
senior = df[df['experience_level'] == 'SE']
print(f"Senior employees: {len(senior)} rows")
senior.head(3)

Senior employees: 42926 rows


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M
1,2025,SE,FT,Data Product Owner,110000,USD,110000,US,0,US,M
2,2025,SE,FT,Data Product Owner,170000,USD,170000,US,0,US,M


In [4]:
# Single condition — numerical
high_salary = df[df['salary_in_usd'] > 150000]
print(f"Salaries above $150k: {len(high_salary)} rows")

Salaries above $150k: 34758 rows


In [5]:
# Use & for AND, | for OR. Wrap each condition in parentheses.
# Combined conditions: AND (both must be true)
premium = df[
    (df['experience_level'] == 'SE') &
    (df['salary_in_usd'] > 150000) &
    (df['remote_ratio'] == 100)
]
print(f"Senior + >$150k + fully remote: {len(premium)} rows")
premium[['job_title', 'experience_level', 'salary_in_usd']].head()

Senior + >$150k + fully remote: 5614 rows


,job_title,experience_level,salary_in_usd
54,Technical Lead,SE,175000
82,Software Engineer,SE,183580
92,Analytics Engineer,SE,154688
108,Data Architect,SE,224381
134,Business Intelligence Lead,SE,161700


> **Why parentheses around each condition?** Python's operator precedence makes `&` and `|` bind more tightly than comparison operators like `==` and `>`. Without parentheses, `df['col'] == 'val' & df['col2'] > 100` is evaluated in the wrong order — `&` runs first, producing incorrect results or a `TypeError`. A related mistake is using Python's `and`/`or` instead of `&`/`|`, which raises `ValueError: The truth value of a Series is ambiguous`. Always wrap each condition in parentheses, and use `&`/`|`, not `and`/`or`.

In [6]:
# Combined conditions: OR (at least one must be true)
executive_or_high_pay = df[
    (df['experience_level'] == 'EX') |
    (df['salary_in_usd'] > 200000)
]
print(f"Executive OR salary >$200k: {len(executive_or_high_pay)} rows")

Executive OR salary >$200k: 18121 rows


In [7]:
# Filter by a list of values with .isin()
ml_roles = df[df['job_title'].isin(['Machine Learning Engineer', 'Data Scientist', 'ML Engineer'])]
print(f"ML-focused roles: {len(ml_roles)} rows")
print(ml_roles['job_title'].value_counts())

ML-focused roles: 17250 rows
job_title
Data Scientist               11443
Machine Learning Engineer     5807
Name: count, dtype: int64


In [8]:
# Negation with ~ (tilde) — invert any boolean mask
# Common pattern: "everything EXCEPT these"
not_ml = df[~df['job_title'].isin(['Machine Learning Engineer', 'Data Scientist', 'ML Engineer'])]
print(f"Non-ML roles: {len(not_ml)} rows")

# Negation equivalent: exclude entries where employment_type is not FT
not_fulltime = df[df['employment_type'] != 'FT']
print(f"Non full-time: {len(not_fulltime)} rows")

Non-ML roles: 55898 rows
Non full-time: 340 rows


<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Best practice: use .loc[mask, columns] for filtering + column selection together</b>

Combine filtering and column selection in one step:

```python
df.loc[df['salary_in_usd'] > 150000, ['job_title', 'salary_in_usd']]
```

This is safer than chained indexing (<code>df[mask]['col']</code>), which can return a copy and produce <code>SettingWithCopyWarning</code> when you try to assign to it.

</div>

**TODO:** Filter for full-time employees (`FT`) working in large companies (`L`) with a salary above $100,000 USD. How many are there?

In [9]:
result = df[
    (df['employment_type'] == 'FT') &
    (df['company_size']== 'L')&
    (df['salary_in_usd'] > 100000)
]
print(f"{len(result)} employees match")

1865 employees match


<details><summary><b>Solution — click to expand</b></summary>

```python
result = df[
    (df['employment_type'] == 'FT') &
    (df['company_size'] == 'L') &
    (df['salary_in_usd'] > 100000)
]
print(f"{len(result)} employees match")
result[['job_title', 'experience_level', 'company_size', 'salary_in_usd']].head()
```
</details>

---
## Step 3: Sorting

`sort_values()` orders rows by one or more columns.

In [10]:
# Sort by salary descending
top_salaries = df.sort_values('salary_in_usd', ascending=False)
top_salaries[['job_title', 'experience_level', 'salary_in_usd']].head(5)

,job_title,experience_level,salary_in_usd
59418,AI Architect,MI,800000
60261,Data Analyst,EN,774000
61236,Machine Learning Scientist,MI,750000
47314,Data Scientist,SE,750000
63026,Machine Learning Engineer,MI,750000


In [ ]:
# Sort by multiple columns
# First by experience level (ascending), then salary (descending) within each level
multi_sort = df.sort_values(
    ['experience_level', 'salary_in_usd'],
    ascending=[True, False]
)
multi_sort[['experience_level', 'job_title', 'salary_in_usd']].head(10)

In [11]:
# nlargest / nsmallest — more efficient than sort_values for top-N
top_10 = df.nlargest(10, 'salary_in_usd')
top_10[['job_title', 'experience_level', 'salary_in_usd']]

,job_title,experience_level,salary_in_usd
59418,AI Architect,MI,800000
60261,Data Analyst,EN,774000
47314,Data Scientist,SE,750000
61060,Analytics Engineer,SE,750000
61236,Machine Learning Scientist,MI,750000
61647,Data Analyst,SE,750000
62422,Machine Learning Scientist,MI,750000
63020,Machine Learning Scientist,MI,750000
63026,Machine Learning Engineer,MI,750000
64004,Data Engineer,MI,750000


---
## Step 4: Groupby & Aggregation

`groupby` follows a **split → apply → combine** pattern: split the rows into groups by a key column, apply an aggregation function to each group independently, then combine the results back into a single table. This is how you answer questions like "what is the average salary per experience level?"

In [12]:
# Average salary by experience level
result = (
    df.groupby('experience_level')
    .agg(avg_salary=('salary_in_usd', 'mean'),
         median_salary=('salary_in_usd', 'median'),
         count=('salary_in_usd', 'count'))
    .reset_index()
    .round(0)
)
result

,experience_level,avg_salary,median_salary,count
0,EN,101701.0,89700.0,6877
1,EX,199244.0,192168.0,1494
2,MI,142892.0,130800.0,21851
3,SE,173298.0,162100.0,42926


> **Note:** `reset_index()` converts the group keys from the index back into regular columns. Always call it after `groupby` to keep the result as a plain DataFrame. Group keys sort alphabetically by default — experience levels above appear as EN/EX/MI/SE, not career order. To sort by value instead, add `.sort_values('avg_salary', ascending=False)`.

In [13]:
# Group by multiple columns and sort by average salary
multi_group = (
    df.groupby(['experience_level', 'company_size'])
    .agg(avg_salary=('salary_in_usd', 'mean'),
         count=('salary_in_usd', 'count'))
    .reset_index()
    .sort_values('avg_salary', ascending=False)
    .round(0)
)
multi_group.head(10)

,experience_level,company_size,avg_salary,count
4,EX,M,199844.0,1461
3,EX,L,173795.0,25
10,SE,M,173416.0,41738
9,SE,L,172700.0,1120
5,EX,S,169172.0,8
6,MI,L,147363.0,987
7,MI,M,142921.0,20791
11,SE,S,110861.0,68
0,EN,L,102504.0,275
1,EN,M,101978.0,6546


<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Best practice: named aggregation in .agg()</b>

Use named aggregation syntax to give your result columns meaningful names:

```python
.agg(avg_salary=('salary_in_usd', 'mean'),
     count=('salary_in_usd', 'count'))
```

This is more readable than <code>.agg({'salary_in_usd': 'mean'})</code> and avoids ambiguous column names when you apply multiple functions.

</div>

**TODO:** Calculate the average and median `salary_in_usd` grouped by `remote_ratio`. Which remote policy pays the highest average salary?

In [14]:
# YOUR TURN
remote_group = (
    df.groupby('remote_ratio')
    .agg(avg_salary=('salary_in_usd', 'mean'),
         median_salary=('salary_in_usd', 'median'))
    .reset_index()
    .sort_values('avg_salary', ascending=False)
    .round(0)
)
remote_group

,remote_ratio,avg_salary,median_salary
0,0,160858.0,149282.0
2,100,148971.0,144300.0
1,50,82361.0,66970.0


<details><summary><b>Solution — click to expand</b></summary>

```python
result = (
    df.groupby('remote_ratio')
    .agg(avg_salary=('salary_in_usd', 'mean'),
         median_salary=('salary_in_usd', 'median'),
         count=('salary_in_usd', 'count'))
    .reset_index()
    .sort_values('avg_salary', ascending=False)
    .round(0)
)
result
```
</details>

---
## Step 5: Creating New Columns

New columns are typically derived from existing ones. Good practice is to work on a copy of the DataFrame to avoid modifying the original.

In [15]:
df_features = df.copy()

# Boolean column: is the employee senior or executive?
df_features['is_senior'] = df_features['experience_level'].isin(['SE', 'EX'])

# Boolean column: fully remote?
df_features['is_fully_remote'] = df_features['remote_ratio'] == 100

# Numerical: salary in thousands (cleaner to read)
df_features['salary_k'] = df_features['salary_in_usd'] / 1000

df_features[['job_title', 'experience_level', 'is_senior', 'is_fully_remote', 'salary_k']].head()

,job_title,experience_level,is_senior,is_fully_remote,salary_k
0,Data Product Owner,SE,True,False,170.0
1,Data Product Owner,SE,True,False,110.0
2,Data Product Owner,SE,True,False,170.0
3,Data Product Owner,SE,True,False,110.0
4,Engineer,SE,True,False,143.0


In [16]:
# pd.cut() bins a continuous column into labeled categories
# bins define the boundaries; labels name each interval
# float('inf') represents positive infinity — the last bin captures any salary above $150k
df_features['salary_tier'] = pd.cut(
    df_features['salary_in_usd'],
    bins=[0, 80000, 150000, float('inf')],
    labels=['low', 'mid', 'high'],
    right=False  # intervals are [low, high): < $80k, $80k-$150k, >= $150k
)

print(df_features['salary_tier'].value_counts())

salary_tier
high    35850
mid     28839
low      8459
Name: count, dtype: int64


In [17]:
# Combine two string columns into one
df_features['role_level'] = df_features['job_title'] + ' (' + df_features['experience_level'] + ')'
print(df_features['role_level'].sample(3, random_state=42))

484      Data Management Specialist (MI)
68397                  Data Analyst (SE)
20934                  Data Analyst (EN)
Name: role_level, dtype: str


<div style="background-color:#1e293b;padding:15px;border-left:6px solid #38bdf8;color:#e2e8f0">

<b>Best practice: use .copy() before modifying a subset</b>

When you create new columns on a slice of a DataFrame, always call <code>.copy()</code> first:

```python
df_subset = df[df['experience_level'] == 'SE'].copy()
df_subset['new_col'] = ...
```

Without <code>.copy()</code>, pandas may return a view of the original. Assignments to a view produce <code>SettingWithCopyWarning</code> and may not behave as expected.

</div>

**TODO:** Add a column called `is_large_company` that is `True` when `company_size == 'L'`, and a column called `recent` that is `True` when `work_year >= 2023`. Print the count of large-company AND recent rows.

In [18]:
# YOUR TURN
df_features['is_large_company'] = df_features['company_size'] == 'L'
df_features['recent'] = df_features['work_year'] >= 2023

count = (df_features['is_large_company'] & df_features['recent']).sum()
print(f"Large company AND recent: {count} rows")

Large company AND recent: 2054 rows


<details><summary><b>Solution — click to expand</b></summary>

```python
df_features['is_large_company'] = df_features['company_size'] == 'L'
df_features['recent'] = df_features['work_year'] >= 2023

count = (df_features['is_large_company'] & df_features['recent']).sum()
print(f"Large company AND recent: {count} rows")
```
</details>

---
## Step 6: Two More Essential Operations

Two operations you will need constantly in real projects: renaming columns and filling missing values.

In [19]:
# .rename() — rename one or more columns
df_renamed = df.rename(columns={
    'salary_in_usd': 'salary_usd',
    'employee_residence': 'residence'
})
print("Renamed columns:", df_renamed.columns.tolist())

Renamed columns: ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_usd', 'residence', 'remote_ratio', 'company_location', 'company_size']


> **Note:** `.rename()` returns a new DataFrame — the original is unchanged. Pass a dict mapping old names to new names. You only need to list the columns you want to rename.
>
> **Series vs DataFrame:** Most operations you've seen — `fillna()`, `sort_values()`, `describe()`, `isnull()`, `rename()`, etc. — work the same way on a single column (Series) as on a full DataFrame. You can apply them directly: `df['salary_in_usd'].fillna(0)` or calling `fillna` on the whole table — both work.

In [20]:
# Create a small example with missing values
sample = df[['job_title', 'experience_level', 'salary_in_usd']].head(5).copy()
sample.loc[2, 'salary_in_usd'] = None  # None is converted to NaN for numeric columns
print("Before fillna:")
print(sample)

# .fillna() — replace NaN with a specified value
median_salary = df['salary_in_usd'].median()
sample['salary_in_usd'] = sample['salary_in_usd'].fillna(median_salary)
print(f"\nAfter fillna (filled with median ${median_salary:,.0f}):")
print(sample)

Before fillna:
            job_title experience_level  salary_in_usd
0  Data Product Owner               SE       170000.0
1  Data Product Owner               SE       110000.0
2  Data Product Owner               SE            NaN
3  Data Product Owner               SE       110000.0
4            Engineer               SE       143000.0

After fillna (filled with median $147,500):
            job_title experience_level  salary_in_usd
0  Data Product Owner               SE       170000.0
1  Data Product Owner               SE       110000.0
2  Data Product Owner               SE       147500.0
3  Data Product Owner               SE       110000.0
4            Engineer               SE       143000.0


> **Note:** Common simple fill strategies: `fillna(median)` or `fillna(mean)` for numerical columns, `fillna(mode)` for categorical columns. The right choice depends on the column and the problem (sometimes this requires more complex logic).

---
## Summary

You can now:
- Filter rows with single and combined boolean conditions using `&`, `|`, `.isin()`, and `~`
- Sort by one or more columns with `sort_values()` and `nlargest()`
- Aggregate groups with `groupby().agg()` using named aggregation + `reset_index()` (split → apply → combine)
- Create derived columns from arithmetic, boolean conditions, and `pd.cut()` for binning
- Work safely on subsets with `.copy()`
- Rename columns with `.rename()`
- Fill missing values with `.fillna()`